# Case 12 -- Layered Coastal Aquifer with a Semiconfining Layer (SHARP verification problem 3)

Steady two-fluid, three-layer comparison with Mualem and Bear (1974), as used to verify
SHARP (Essaid, 1990, USGS WRIR 90-4130, figures 15-17 and table 3). Sea on the left,
steady freshwater inflow on the right, and a thin low-permeability layer of finite
length part way down that splits the interface into a branch above the layer and a
branch below it. This is the only benchmark with a split interface across a leaky bed,
so it exercises vertical exchange between layers and cells that are entirely one fluid.

In [ ]:
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

# path to mf6 executables with swi support:
#   https://github.com/christianlangevin/modflow6-nightly-build/actions/workflows/nightly-build-swi.yml

# Put the name of the mf6 executable into mf6exe.txt,
# which is not under version control.
with open(pl.Path("./mf6exe.txt"), "r") as f:
    mf6exe = f.readline().strip()
print(f"using executable: {mf6exe}")

sim_ws = pl.Path("./temp/case12")

## Parameters

SHARP table 3 and figure 16, in metres and seconds. The freshwater inflow is given as
1 cm$^3$/s but the apparatus width is not. The no-layer interface in figure 17 is the
Glover solution $z = [2qx/(K\delta)]^{1/2}$, which reaches 30 cm depth at $x \approx 113$ cm
only for $q = 10^{-4}$ m$^2$/s, so the width is 1 cm.

In [ ]:
rhof = 1000.0
rhos = 1025.0
alphaf = rhof / (rhos - rhof)      # 40.0
alphas = rhos / (rhos - rhof)      # 41.0
delta = (rhos - rhof) / rhof       # 0.025

# geometry, SHARP figure 16 (metres)
Lx = 1.15                          # domain length; frame edge of fig. 17 is ~115 cm
b_upper, b_conf, b_lower = 0.15, 0.01, 0.24
top = 0.0
botm = [-b_upper, -(b_upper + b_conf), -(b_upper + b_conf + b_lower)]
thick = np.array([b_upper, b_conf, b_lower])
conf_x0, conf_x1 = 0.10, 0.80      # extent of the semiconfining layer, digitized

# material properties, SHARP table 3
kaq = 0.10                         # Kf, m/s
kconf = 8.0e-4                     # K', m/s
n = 0.1                            # porosity -> interface storage (SY)

width = 0.01                       # apparatus width, recovered above
Qin = 1.0e-6                       # 1 cm3/s
q = Qin / width                    # 1e-4 m2/s, discharge per unit width

dx_sharp = 0.05                    # SHARP table 3
dx_fine = 0.0125


def cell_centers(dx):
    ncol = int(round(Lx / dx))
    return np.linspace(0.5 * dx, Lx - 0.5 * dx, ncol)


def glover(xx):
    """Interface depth with no semiconfining layer (Glover, 1959; no outflow face).

    This is the dashed line in SHARP figure 17.
    """
    return np.sqrt(2.0 * q * xx / (kaq * delta))


for xq in (0.40, 0.80, 1.13):
    print(f"no-layer interface at x = {100 * xq:5.0f} cm : "
          f"{100 * glover(xq):5.1f} cm depth")

## Reference points

Digitized from SHARP figure 17A (restricted mixing), which plots the analytical solution
and SHARP's points. Mualem and Bear (1974) is not in `ref/`, so the analytical curve
itself is not evaluated. Expect about 0.2 cm of digitizing error.

In [ ]:
# (x, depth) in cm, digitized from SHARP figure 17A (restricted mixing, method 2)
sharp_upper = np.array([
    [6.21, 6.06], [11.12, 8.76], [15.85, 10.66], [20.57, 12.26], [25.54, 13.78],
])
sharp_lower = np.array([
    [12.33, 16.05], [16.21, 16.42], [21.36, 17.04], [26.50, 17.72], [31.10, 18.32],
    [36.33, 18.99], [41.33, 19.73], [46.67, 20.40], [51.79, 21.09], [56.27, 21.79],
    [61.90, 22.53], [66.84, 23.20], [71.27, 23.82], [76.76, 24.62], [81.40, 25.20],
    [86.12, 25.91], [101.52, 27.82], [106.54, 28.46],
])
print(f"{len(sharp_upper)} points above the layer, {len(sharp_lower)} below")

## Build the model

Three layers: upper aquifer (15 cm), semiconfining layer (1 cm), lower aquifer (24 cm).
SHARP represents the bed as a leakance; here it is an explicit layer, which adds the
aquifer half-thicknesses to the vertical resistance (about 15 percent more than SHARP).
Boundary conditions follow figure 16: zero freshwater and saltwater head at the sea, and
the inflow distributed over the inland face by layer thickness. The steady state is
reached with a transient run and growing time steps, checked for stationarity below.

In [ ]:
def kfield(x, semiconfining=True):
    k = np.full((3, 1, x.size), kaq)
    if semiconfining:
        k[1, 0, (x >= conf_x0) & (x <= conf_x1)] = kconf
    return k


def build_model(ws, dx=dx_fine, semiconfining=True, two_fluid=True,
                perlen=20000.0, nstp=150):
    x = cell_centers(dx)
    ncol = x.size
    k = kfield(x, semiconfining)

    sim = flopy.mf6.MFSimulation(sim_name="layered", sim_ws=ws, exe_name=mf6exe)
    flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[(perlen, nstp, 1.05)],
                          time_units="seconds")
    ims = flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        no_ptcrecord=True,
        outer_maximum=500,
        inner_maximum=200,
        outer_dvclose=1.0e-9,
        inner_dvclose=1.0e-10,
        linear_acceleration="bicgstab",
        backtracking_number=20,
        backtracking_tolerance=1.05,
        backtracking_reduction_factor=0.1,
        backtracking_residual_limit=0.002,
    )

    # initial guess: Ghyben-Herzberg applied to the no-layer solution
    zeta0 = -np.minimum(glover(x), -botm[-1])
    hf0 = np.tile(-zeta0 / alphaf, (3, 1, 1))

    models = (False, True) if two_fluid else (False,)
    for is_saltwater in models:
        name = "saltwater" if is_saltwater else "freshwater"
        gwf = flopy.mf6.ModflowGwf(sim, modelname=name, save_flows=True,
                                   newtonoptions="NEWTON")
        flopy.mf6.ModflowGwfdis(gwf, nlay=3, nrow=1, ncol=ncol, delr=dx,
                                delc=width, top=top, botm=botm)
        flopy.mf6.ModflowGwfic(
            gwf, strt=np.zeros((3, 1, ncol)) if is_saltwater else hf0)
        flopy.mf6.ModflowGwfnpf(gwf, icelltype=0, k=k, k33=k,
                                save_specific_discharge=True,
                                save_saturation=True)
        flopy.mf6.ModflowGwfsto(gwf, iconvert=0, ss=0.0, sy=n,
                                transient={0: True})
        flopy.mf6.ModflowGwfswi(gwf, zeta_filerecord=f"{name}.zta")
        # sea boundary: freshwater and saltwater head both zero
        flopy.mf6.ModflowGwfchd(
            gwf, stress_period_data=[[kk, 0, 0, 0.0] for kk in range(3)])
        if not is_saltwater:
            # inland freshwater inflow, distributed by layer thickness
            flopy.mf6.ModflowGwfwel(
                gwf,
                stress_period_data=[[kk, 0, ncol - 1, Qin * thick[kk] / thick.sum()]
                                    for kk in range(3)])
        flopy.mf6.ModflowGwfoc(
            gwf, budget_filerecord=f"{name}.bud", head_filerecord=f"{name}.hds",
            saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")])

    if two_fluid:
        flopy.mf6.ModflowSwiswi(sim, exgtype="SWI6-SWI6",
                                exgmnamea="freshwater", exgmnameb="saltwater")
        sim.register_ims_package(ims, ["freshwater", "saltwater"])
    return sim


def run(ws, **kwargs):
    sim = build_model(ws, **kwargs)
    sim.write_simulation(silent=True)
    ok, buff = sim.run_simulation(silent=True)
    if not ok:
        print("\n".join(buff[-30:]))
        raise RuntimeError(f"{ws} did not converge")
    dx = kwargs.get("dx", dx_fine)
    x = cell_centers(dx)
    zobj = flopy.utils.HeadFile(pl.Path(ws) / "freshwater.zta", text="zeta")
    z = zobj.get_alldata().reshape(-1, 3, x.size)
    drift = np.abs(z[-1] - z[-2]).max()
    return x, z[-1], drift

In [ ]:
x, zeta, drift = run(sim_ws / "layered", dx=dx_fine, semiconfining=True)
xc, zeta_nolayer, drift_nl = run(sim_ws / "nolayer", dx=dx_fine, semiconfining=False)
print(f"layered   : max |d zeta| over the last time step = {drift:.2e} m")
print(f"no-layer  : max |d zeta| over the last time step = {drift_nl:.2e} m")

bud = flopy.utils.CellBudgetFile(sim_ws / "layered" / "freshwater.bud")
qwel = sum(r[2] for r in bud.get_data(text="WEL")[-1])
qchd = sum(r[2] for r in bud.get_data(text="CHD")[-1])
print(f"freshwater budget: in {qwel:.4e}, out {qchd:.4e} m3/s "
      f"(imbalance {100 * abs(qwel + qchd) / qwel:.2e} %)")

## Compare with SHARP figure 17A

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.add_patch(Rectangle((100 * conf_x0, -100 * botm[0]),
                       100 * (conf_x1 - conf_x0), 100 * b_conf,
                       facecolor="0.75", edgecolor="k", hatch="///", zorder=1,
                       label="semiconfining layer"))

xf = np.linspace(0.001, Lx, 400)
ax.plot(100 * xf, 100 * glover(xf), "k--", lw=1.2,
        label="analytical, no semiconfining layer")

upper = np.ma.masked_where(zeta[0] <= botm[0] + 1e-6, -100 * zeta[0])
lower = np.ma.masked_where(zeta[2] >= botm[1] - 1e-6, -100 * zeta[2])
ax.plot(100 * x, upper, "-", color="tab:red", lw=2, label="SWI, above the layer")
ax.plot(100 * x, lower, "-", color="tab:blue", lw=2, label="SWI, below the layer")

ax.plot(sharp_upper[:, 0], sharp_upper[:, 1], "o", ms=6, mfc="none",
        mec="k", mew=1.2, label="SHARP fig. 17A (digitized)")
ax.plot(sharp_lower[:, 0], sharp_lower[:, 1], "o", ms=6, mfc="none",
        mec="k", mew=1.2)

ax.set_xlim(0, 100 * Lx)
ax.set_ylim(100 * -botm[-1], 0)
ax.set_xlabel("DISTANCE (cm)")
ax.set_ylabel("DEPTH (cm)")
ax.set_title("Layered coastal aquifer, restricted mixing (compare SHARP fig. 17A)")
ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()

## Quantitative comparison

In [ ]:
def compare(x, zeta, label):
    du = np.array([-100 * np.interp(p[0] / 100, x, zeta[0]) - p[1]
                   for p in sharp_upper])
    dl = np.array([-100 * np.interp(p[0] / 100, x, zeta[2]) - p[1]
                   for p in sharp_lower])
    print(f"{label:>26}  upper branch: mean {du.mean():+5.2f} cm, "
          f"max |diff| {np.abs(du).max():4.2f} cm")
    print(f"{'':>26}  lower branch: mean {dl.mean():+5.2f} cm, "
          f"max |diff| {np.abs(dl).max():4.2f} cm")
    return du, dl


def meets_layer(x, zeta):
    """x where the upper-aquifer interface reaches the top of the layer."""
    up = zeta[0]
    j = int(np.argmax(up <= botm[0] + 1e-5))
    if j == 0:
        return np.nan
    return np.interp(botm[0], up[:j + 1][::-1], x[:j + 1][::-1])


compare(x, zeta, f"dx = {dx_fine} m")
print(f"\nupper interface meets the semiconfining layer at "
      f"x = {100 * meets_layer(x, zeta):.1f} cm")
print("SHARP fig. 17A shows it meeting the layer at about 29 cm")

## Grid refinement

The no-layer control has a closed-form answer; the layered case is compared with SHARP
at SHARP's grid spacing and at a refined one. The interface is steep near the shoreline,
where SHARP resolves the tip with its tracking algorithm and SWI needs the grid.

In [ ]:
print("no-layer control against the analytical solution (depth, cm)\n")
hdr = f"{'x (cm)':>8}"
runs = {}
for dxr in (0.05, 0.025, 0.0125):
    runs[dxr] = run(sim_ws / f"nl_{dxr}", dx=dxr, semiconfining=False)
    hdr += f"{'dx=' + str(dxr):>11}"
print(hdr + f"{'analytical':>12}")

for xq in (0.075, 0.125, 0.225, 0.325, 0.525, 0.725, 0.925, 1.125):
    row = f"{100 * xq:8.1f}"
    for dxr in (0.05, 0.025, 0.0125):
        xr, zr, _ = runs[dxr]
        deep = np.where(zr[0] > botm[0], zr[0], zr[2])
        row += f"{-100 * np.interp(xq, xr, deep):11.2f}"
    print(row + f"{100 * glover(xq):12.2f}")

In [ ]:
print("layered case against SHARP figure 17A\n")
for dxr in (dx_sharp, dx_fine):
    xr, zr, _ = run(sim_ws / f"lay_{dxr}", dx=dxr, semiconfining=True)
    compare(xr, zr, f"dx = {dxr} m")
    print(f"{'':>26}  meets the layer at x = {100 * meets_layer(xr, zr):.1f} cm\n")

## Effect of the buoyancy restriction

At steady state the saltwater is static (its only boundary is the sea), so the two-fluid
and single-fluid runs differ only in whether vertical flow is restricted.

In [ ]:
x2, zeta2, _ = run(sim_ws / "twofluid", dx=dx_fine, semiconfining=True,
                   two_fluid=True)
x1, zeta1, _ = run(sim_ws / "onefluid", dx=dx_fine, semiconfining=True,
                   two_fluid=False)

hs = flopy.utils.HeadFile(sim_ws / "twofluid" / "saltwater.hds").get_alldata()[-1]
print(f"two-fluid saltwater head at steady state: "
      f"min {hs.min():.2e} m, max {hs.max():.2e} m  (static)\n")

print(f"{'x (cm)':>8} {'2-fluid up':>11} {'1-fluid up':>11} "
      f"{'2-fluid low':>12} {'1-fluid low':>12} {'diff':>8}")
worst = 0.0
for xq in (0.10, 0.20, 0.30, 0.40, 0.60, 0.80, 1.00, 1.125):
    j = int(np.argmin(np.abs(x2 - xq)))
    d = max(abs(zeta2[0, j] - zeta1[0, j]), abs(zeta2[2, j] - zeta1[2, j]))
    worst = max(worst, d)
    print(f"{100 * x2[j]:8.1f} {-100 * zeta2[0, j]:11.2f} {-100 * zeta1[0, j]:11.2f} "
          f"{-100 * zeta2[2, j]:12.2f} {-100 * zeta1[2, j]:12.2f} {100 * d:8.3f}")
print(f"\nlargest difference anywhere: {100 * np.abs(zeta2 - zeta1).max():.3f} cm")

## Notes

- On a 1.25 cm grid the interface matches SHARP figure 17A to a mean of 0.3 cm above the
  layer and 0.5 cm below, on a 40 cm aquifer. The freshwater budget closes to 0.1 percent.
- Grid resolution matters more than for SHARP. On SHARP's 5 cm grid the upper branch is
  2 cm too shallow and meets the layer at 37 cm instead of 29 cm; at 1.25 cm the error is
  0.3 cm and the meeting point 33 cm. The lower branch starts about 8 cm further inland
  than SHARP's for the same reason. Refine the grid near pinch-outs when translating
  SHARP or SWI2 problems.
- The buoyancy restriction is not exercised here: two-fluid and single-fluid agree to
  0.08 cm. Leakage through the bed is upward freshwater, which is unrestricted. Case 11
  is the contrast.
- SWI conserves both fluids and corresponds to SHARP's restricted-mixing option (method
  2, figure 17A). It has no analog of complete mixing (method 1, figure 17B), in which
  leaked freshwater is converted to saltwater.
- Inferred rather than read from the report: the apparatus width (1 cm), the horizontal
  extent of the layer (10-80 cm), and the explicit-layer treatment of the bed.